In [2]:
import nflreadpy
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error
from scipy.stats import pearsonr
import warnings
warnings.filterwarnings('ignore')

# ---- Load current model predictions ----
preds_full = pd.read_pickle("PickleFiles/Full PPR Rankings.pkl")
preds_half = pd.read_pickle("PickleFiles/Half PPR Rankings.pkl")
preds_std  = pd.read_pickle("PickleFiles/Non PPR Rankings.pkl")

# ---- Fetch 2025 actual stats ----
print("Fetching 2025 seasonal data from nflreadpy...")
# nflreadpy player stats already includes player_name and position — no roster merge needed
seasonal_2025 = nflreadpy.load_player_stats([2025], summary_level='reg').to_pandas()
stats = seasonal_2025[seasonal_2025['position'].isin(['QB','RB','WR','TE'])].copy()

# fantasy_points = standard (non-PPR); fantasy_points_ppr = Full PPR
stats['actual_ppr']  = stats['fantasy_points_ppr'] / stats['games']
stats['actual_half'] = (stats['fantasy_points'] + stats['receptions'] * 0.5) / stats['games']
stats['actual_std']  = stats['fantasy_points'] / stats['games']

# Keep players with >= 8 games (avoids injury-shortened samples)
stats = stats[stats['games'] >= 8].copy()
stats = stats.rename(columns={'player_name': 'Name'})

print(f"2025 actuals: {len(stats)} players (>= 8 games played)")
print(f"Full PPR predictions: {len(preds_full)} players")


Fetching 2024 seasonal data from nfl_data_py...
2024 actuals: 333 players (>= 8 games played)
Full PPR predictions: 285 players


In [3]:
def evaluate(preds_df, actuals_df, actual_col, scoring_name):
    merged = preds_df[['Name','Position','Final PPG']].merge(
        actuals_df[['Name', actual_col, 'position']], 
        on='Name', how='inner'
    )
    if len(merged) < 10:
        print(f"{scoring_name}: only {len(merged)} name matches — check name format")
        return None
    
    mae  = mean_absolute_error(merged[actual_col], merged['Final PPG'])
    rmse = np.sqrt(((merged[actual_col] - merged['Final PPG'])**2).mean())
    corr, pval = pearsonr(merged['Final PPG'], merged[actual_col])
    
    merged['Error']     = merged['Final PPG'] - merged[actual_col]
    merged['Abs_Error'] = merged['Error'].abs()
    
    print("\n" + "="*55)
    print(f" {scoring_name}")
    print("="*55)
    print(f"  Matched players: {len(merged)}")
    print(f"  MAE:             {mae:.2f} PPG")
    print(f"  RMSE:            {rmse:.2f} PPG")
    print(f"  Correlation:     {corr:.3f}  (p={pval:.4f})")
    
    print("\n  By position (players with >= 8 games):")
    for pos in ['QB','RB','WR','TE']:
        sub = merged[merged['position'] == pos]
        if len(sub) >= 5:
            pm  = mean_absolute_error(sub[actual_col], sub['Final PPG'])
            pc, _ = pearsonr(sub['Final PPG'], sub[actual_col])
            print(f"    {pos}: n={len(sub):3d} | MAE={pm:.2f} | r={pc:.3f}")
    
    print("\n  10 worst predictions (highest absolute error):")
    worst = merged.nlargest(10, 'Abs_Error')[['Name','position','Final PPG',actual_col,'Error']]
    worst.columns = ['Name','Pos','Pred','Actual','Error']
    print(worst.to_string(index=False))
    
    return merged

eval_full = evaluate(preds_full, stats, 'actual_ppr',  'FULL PPR — Current Model')
eval_half = evaluate(preds_half, stats, 'actual_half', 'HALF PPR — Current Model')
eval_std  = evaluate(preds_std,  stats, 'actual_std',  'STANDARD — Current Model')


 FULL PPR — Current Model
  Matched players: 218
  MAE:             4.22 PPG
  RMSE:            5.61 PPG
  Correlation:     0.242  (p=0.0003)

  By position (players with >= 8 games):
    QB: n= 24 | MAE=5.76 | r=-0.053
    RB: n= 55 | MAE=4.46 | r=0.119
    WR: n= 87 | MAE=3.88 | r=-0.074
    TE: n= 52 | MAE=3.80 | r=0.018

  10 worst predictions (highest absolute error):
          Name Pos      Pred    Actual      Error
Saquon Barkley  RB  2.511600 20.143750 -17.632150
 Derrick Henry  RB  2.195004 18.670588 -16.475584
 Lamar Jackson  QB  8.996358 25.316471 -16.320113
Elijah Higgins  TE 17.222584  2.085714  15.136869
   Trey Palmer  WR 16.913191  2.320000  14.593191
    Josh Allen  QB  9.162870 23.271250 -14.108380
   Jalin Hyatt  WR 14.677705  0.620000  14.057705
Jonathan Mingo  WR 14.606965  1.085714  13.521251
     Jake Bobo  WR 14.216312  1.518182  12.698131
Hassan Haskins  RB 15.264232  2.709091  12.555141

 HALF PPR — Current Model
  Matched players: 218
  MAE:             3.70

In [4]:
print("\n" + "="*55)
print(" BASELINE SUMMARY — Current Model vs 2024 Actuals")
print("="*55)
print(f"{'Format':<15} {'Matched':>8} {'MAE (PPG)':>12} {'Correlation':>13}")
print("-"*55)
for name, ev, col in [('Full PPR', eval_full, 'actual_ppr'),
                       ('Half PPR', eval_half, 'actual_half'),
                       ('Standard', eval_std,  'actual_std')]:
    if ev is not None and len(ev) > 5:
        m = mean_absolute_error(ev[col], ev['Final PPG'])
        c, _ = pearsonr(ev['Final PPG'], ev[col])
        print(f"{name:<15} {len(ev):>8} {m:>12.2f} {c:>13.3f}")
    else:
        print(f"{name:<15}    N/A")
print()
print("Run ImprovedMLModel.ipynb to retrain and compare.")


 BASELINE SUMMARY — Current Model vs 2024 Actuals
Format           Matched    MAE (PPG)   Correlation
-------------------------------------------------------
Full PPR             218         4.22         0.242
Half PPR             218         3.70         0.414
Standard             218         3.28         0.585

Run ImprovedMLModel.ipynb to retrain and compare.
